# 제공 데이터 탐색 (EDA)

가이드 4-4 탐색 항목: 구성 · 기간 · 길이 · 품질 · 내용

대상: `data/raw/02_AI반도체_묶음_*.csv` (키워드별 파일 2개의 합본)

In [ ]:
import glob, re
import pandas as pd

# 프로젝트 루트 기준 경로 (notebooks/ 에서 실행)
f = glob.glob('../data/raw/*.csv')[0]
df = pd.read_csv(f, encoding='utf-8-sig')
df['본문전문'] = df['본문전문'].fillna('')
df['d'] = pd.to_datetime(df['발행일'])
print(df.shape)
df.head(3)

## 1. 구성 — 결측, 문서 종류, 수집 키워드, 매체

In [ ]:
print(df.isna().sum())
print(df['문서종류'].value_counts())
print(df['수집키워드'].value_counts())
print('매체 수:', df['매체명'].nunique())
df['매체명'].value_counts().head(15)

## 2. 기간 — 연도·분기별 분포

In [ ]:
print(df['d'].min(), df['d'].max())
print(df['d'].dt.year.value_counts().sort_index())
df.groupby(df['d'].dt.to_period('Q')).size()

## 3. 길이 — 본문 길이 분포, 과소·과다 문서

In [ ]:
L = df['본문전문'].str.len()
print(L.describe(percentiles=[.05,.25,.5,.75,.95]))
print('200자 미만:', (L<200).sum(), '/ 5000자 초과:', (L>5000).sum())

## 4. 품질 — 중복, 보일러플레이트, 관련성

In [ ]:
# 중복 추정 (정밀 판별은 indexer/dedup.py)
def norm_title(t):
    return re.sub(r'\[[^\]]*\]|\([^)]*\)|[^가-힣A-Za-z0-9]', '', str(t)).lower()
df['nt'] = df['제목'].map(norm_title)
df['head'] = df['본문전문'].str.replace(r'\s+', '', regex=True).str[:80]
print('URL 중복:', df['원문URL'].duplicated().sum())
print('본문 완전 중복:', df['본문전문'].duplicated().sum())
print('정규화 제목 or 본문 앞 80자 중복:', (df['nt'].duplicated() | df['head'].duplicated()).sum())

In [ ]:
# 기사 말미 보일러플레이트
for p in ['기사 섹션 분류 안내', '이 기사를 추천합니다', '기사를 더 보고 싶다면', '무단전재']:
    print(p, df['본문전문'].str.contains(p, regex=False).sum())

In [ ]:
# 관련성: 본문 내 핵심 키워드 등장 횟수
c = df['본문전문'].str.count(r'AI ?반도체|인공지능 ?반도체')
print(c.clip(upper=5).value_counts().sort_index())
t = df['제목'].str.contains('반도체|HBM|NPU|칩|AI 가속기|퓨리오사|리벨리온|딥엑스|모빌린트')  # collector/load_provided.py 와 동일
rel = t | (c >= 2)
print('관련성 필터:', rel.sum())
print('+ 2024 이후:', (rel & (df['d'] >= '2024-01-01')).sum())
print('+ 2025 이후:', (rel & (df['d'] >= '2025-01-01')).sum())

In [ ]:
# 대표 질문 관련 키워드 커버리지
kw = {'HBM':'HBM','NPU':'NPU','온디바이스':'온디바이스','가속기':'가속기','삼성전자':'삼성전자',
      'SK하이닉스':'SK하이닉스','엔비디아':'엔비디아','한미반도체':'한미반도체','보조금':'보조금',
      '세액공제':'세액공제','수출규제':'수출 ?규제|수출통제','반도체특별법':'반도체 ?특별법','산업연구원':'산업연구원'}
pd.Series({k: df['본문전문'].str.contains(p, regex=True).sum() for k, p in kw.items()})

## 5. 내용 — 샘플 직접 읽기

In [ ]:
for i in df.sample(3, random_state=7).index:
    print(df.at[i,'발행일'], df.at[i,'매체명'], df.at[i,'제목'])
    print(df.at[i,'본문전문'][:500], '\n---')

## 정리 (1차)

- 120,057건, 본문 결측 86건, 문서 종류는 전부 news
- 2024년 이후 비중 87% (최근 편중)
- 본문에 'AI 반도체'가 0회 등장하는 문서 50% → 관련성 필터 필요
- 제목/본문 앞부분 기준 중복 약 8,400건 (재배포본 포함 시 증가 예상)
- 기사 말미 보일러플레이트 존재 → indexer/parse.py 에서 제거